# SIMBA Optimizer

This notebook demonstrates the SIMBA (Self-Improving Model-Based Adaptation) optimizer in DSPy 3.x.

**What You'll Learn:**
- How SIMBA works (self-reflective improvement)
- How to configure SIMBA parameters
- How SIMBA differs from GEPA, BootstrapFewShot, and MIPROv2
- When to use SIMBA vs other optimizers
- A production-ready optimization template

## Setup

In [ ]:
import os
import sys
sys.path.append('../../')

import dspy
from utils import setup_default_lm, configure_dspy, print_step, print_result, print_error
from utils.datasets import get_sample_qa_data
from dotenv import load_dotenv

load_dotenv('../../.env')

try:
    lm = setup_default_lm(provider="openai", model="gpt-4o", max_tokens=500)
    configure_dspy(lm=lm)
    print_result("Language model configured successfully!", "Status")
except Exception as e:
    print_error(f"Failed to configure language model: {e}")

## Part 1: Understanding SIMBA

SIMBA works by iteratively:
1. Running the current program on training examples
2. Identifying failures and analyzing error patterns
3. Generating candidate improvements (instructions, demos)
4. Evaluating candidates and keeping the best
5. Repeating for multiple improvement steps

In [ ]:
print("SIMBA key characteristics:")
print("  - Iterative self-improvement with reflection")
print("  - Optimizes both instructions and few-shot demos")
print("  - Uses the LM itself to propose improvements")
print("  - Batch-based evaluation for efficiency")
print("  - Learns specifically from failure cases")

## Part 2: Basic SIMBA Setup

Prepare a QA system and test its baseline performance.

In [ ]:
class QuestionAnswering(dspy.Signature):
    """Answer questions accurately and concisely."""
    question = dspy.InputField(desc="A question to answer")
    answer = dspy.OutputField(desc="A concise and accurate answer")

class QASystem(dspy.Module):
    def __init__(self):
        super().__init__()
        self.qa = dspy.ChainOfThought(QuestionAnswering)

    def forward(self, question):
        result = self.qa(question=question)
        return dspy.Prediction(answer=result.answer, reasoning=result.reasoning)

all_data = get_sample_qa_data()
train_data = all_data[:8]
val_data = all_data[8:]
print(f"Training examples: {len(train_data)}")
print(f"Validation examples: {len(val_data)}")

In [ ]:
def qa_metric(example, prediction, trace=None):
    pred_answer = prediction.answer.lower().strip()
    true_answer = example.answer.lower().strip()
    if pred_answer == true_answer:
        return 1.0
    true_words = set(true_answer.split())
    pred_words = set(pred_answer.split())
    overlap = len(true_words & pred_words)
    if overlap > 0:
        return overlap / max(len(true_words), len(pred_words))
    return 0.0

print("Testing baseline performance...")
baseline = QASystem()
baseline_scores = []
for example in val_data[:3]:
    try:
        prediction = baseline(question=example.question)
        score = qa_metric(example, prediction)
        baseline_scores.append(score)
        print(f"  Q: {example.question[:60]}...")
        print(f"  Expected: {example.answer}")
        print(f"  Got: {prediction.answer}")
        print(f"  Score: {score:.2f}\n")
    except Exception as e:
        print(f"  Error: {e}")
        baseline_scores.append(0.0)

avg_baseline = sum(baseline_scores) / len(baseline_scores) if baseline_scores else 0
print(f"Baseline average score: {avg_baseline:.2f}")

## Part 3: SIMBA Configuration

Key parameters for tuning SIMBA optimization.

In [ ]:
print("Key SIMBA Parameters:")
print()
print("1. bsize (default: 32)")
print("   - Batch size for evaluation")
print("   - Recommended: 16-64")
print()
print("2. num_candidates (default: 6)")
print("   - Number of improvement candidates per step")
print("   - Recommended: 4-10")
print()
print("3. max_steps (default: 8)")
print("   - Maximum improvement iterations")
print("   - Recommended: 5-15")
print()
print("4. max_demos (default: 4)")
print("   - Maximum few-shot demonstrations")
print("   - Recommended: 2-8")
print()
print("5. num_threads (default: None)")
print("   - Parallel evaluation threads")
print("   - Set to 4-8 for faster optimization")
print()
print("6. temperature_for_sampling (default: 0.2)")
print("   - Temperature for generating candidate improvements")
print("   - Higher = more diverse candidates")

## Part 4: Running SIMBA Optimization

In [ ]:
try:
    print("Initializing SIMBA optimizer...")

    simba_optimizer = dspy.SIMBA(
        metric=qa_metric,
        bsize=8,
        num_candidates=3,
        max_steps=2,
        max_demos=3,
        num_threads=1
    )

    print("\nSIMBA optimization process:")
    print("  1. Evaluate current program on training batch")
    print("  2. Identify failure cases")
    print("  3. Generate candidate improvements")
    print("  4. Evaluate candidates and keep the best")
    print("  5. Repeat for max_steps iterations")

    # Uncomment to run:
    # optimized = simba_optimizer.compile(student=QASystem(), trainset=train_data)

    print("\n[Optimization code is commented out for demo purposes]")
    print("SIMBA typically improves scores through iterative refinement.")

except Exception as e:
    print_error(f"SIMBA error: {e}")

## Part 5: Choosing the Right Optimizer

In [ ]:
print("SIMBA - Iterative refinement with self-reflection")
print("  Best with: 20-100 examples, moderate optimization time")
print()
print("GEPA - Multi-objective genetic optimization")
print("  Best with: 50+ examples, complex multi-goal tasks")
print()
print("BootstrapFewShot - Quick few-shot example selection")
print("  Best with: <20 examples, fast prototyping")
print()
print("MIPROv2 - Balanced Bayesian optimization")
print("  Best with: 20-50 examples, good speed/quality balance")
print()
print("Recommended path:")
print("  1. Start with BootstrapFewShot (quick baseline)")
print("  2. Try SIMBA for iterative improvement")
print("  3. Use GEPA for maximum performance")

## Part 6: Production Template

In [ ]:
print("""
# Production SIMBA optimization template

optimizer = dspy.SIMBA(
    metric=my_metric,
    bsize=32,
    num_candidates=6,
    max_steps=8,
    max_demos=4,
    num_threads=4
)

optimized = optimizer.compile(
    student=MyModule(),
    trainset=train_data
)

# Evaluate
scores = [my_metric(ex, optimized(**ex.inputs())) for ex in val_data]
print(f"Average score: {sum(scores) / len(scores):.3f}")

# Save
optimized.save("models/my_simba_optimized_model")
""")

## Summary

**Key Takeaways:**
1. SIMBA uses self-reflective improvement to optimize programs
2. It learns from failures and iteratively proposes better prompts
3. Key parameters: `bsize`, `num_candidates`, `max_steps`, `max_demos`
4. Works well with moderate training data (20-100 examples)
5. Best when the task benefits from iterative prompt refinement
6. Complements GEPA — SIMBA for refinement, GEPA for exploration
7. Start with BootstrapFewShot, then try SIMBA or GEPA for more improvement